In [4]:
import pandas as pd

In [5]:
df = pd.read_csv(r"datasets\hhblock_dataset\hhblock_dataset\block_0.csv")

In [7]:
# 1. Basic shape and structure
print(df.shape)
print(df.head())
print(df.dtypes)

(25286, 50)
       LCLid         day   hh_0   hh_1   hh_2   hh_3   hh_4   hh_5   hh_6  \
0  MAC000002  2012-10-13  0.263  0.269  0.275  0.256  0.211  0.136  0.161   
1  MAC000002  2012-10-14  0.262  0.166  0.226  0.088  0.126  0.082  0.123   
2  MAC000002  2012-10-15  0.192  0.097  0.141  0.083  0.132  0.070  0.130   
3  MAC000002  2012-10-16  0.237  0.237  0.193  0.118  0.098  0.107  0.094   
4  MAC000002  2012-10-17  0.157  0.211  0.155  0.169  0.101  0.117  0.084   

    hh_7  ...  hh_38  hh_39  hh_40  hh_41  hh_42  hh_43  hh_44  hh_45  hh_46  \
0  0.119  ...  0.918  0.278  0.267  0.239  0.230  0.233  0.235  0.188  0.259   
1  0.083  ...  1.075  0.956  0.821  0.745  0.712  0.511  0.231  0.210  0.278   
2  0.074  ...  1.164  0.249  0.225  0.258  0.260  0.334  0.299  0.236  0.241   
3  0.109  ...  0.966  0.172  0.192  0.228  0.203  0.211  0.188  0.213  0.157   
4  0.118  ...  0.223  0.075  0.230  0.208  0.265  0.377  0.327  0.277  0.288   

   hh_47  
0  0.250  
1  0.159  
2  0.237  


In [8]:
# 2. Missing values — check both explicit NaNs and the string 'Null'
# This dataset is known to sometimes store missing readings as the literal string "Null"
print(df.isnull().sum().sum())  # explicit NaNs across all columns
print((df == 'Null').sum().sum())  # string-based nulls, a known quirk of this dataset

50
0


In [10]:
print(df.groupby('LCLid')['day'].count().describe())


count     50.000000
mean     505.720000
std       91.007522
min      222.000000
25%      489.500000
50%      500.000000
75%      512.500000
max      814.000000
Name: day, dtype: float64


In [11]:
hh_cols = [c for c in df.columns if c.startswith('hh_')]
for col in hh_cols:
    non_numeric = pd.to_numeric(df[col], errors='coerce').isnull().sum()
    if non_numeric > 0:
        print(col, non_numeric)

hh_25 1
hh_30 49


In [12]:
df['day'] = pd.to_datetime(df['day'])
for lclid, group in df.groupby('LCLid'):
    full_range = pd.date_range(group['day'].min(), group['day'].max(), freq='D')
    missing_days = full_range.difference(group['day'])
    if len(missing_days) > 0:
        print(lclid, len(missing_days), "missing days")
        break 

MAC000002 5 missing days


In [13]:
total_cells = df.shape[0] * len(hh_cols)
print(f"{50 / total_cells * 100:.4f}% missing")

0.0041% missing
